# Medical Symptom Classification Through Audio Processing

This notebook presents a systematic workflow for classifying medical symptoms from patient audio data using machine learning and deep learning. The approach mirrors the structure and rigor of the text classification notebook, ensuring research reproducibility and clarity.

## 1. Research Context and Objectives

**Research Question:**  
How effective is NLP and deep learning in classifying patient symptoms from audio data on the population level?

**Hypotheses:**  
- **H20 (Null):** Audio analysis of patient symptoms yields both precision and recall metrics that are insufficient for effective provider decision support.  
- **H2a (Alternative):** Audio analysis of patient symptoms results in precision and recall sufficient for provider decision support.

**Significance:**  
Automated audio-based symptom classification can:
- Capture vocal biomarkers not present in text
- Support patients with limited literacy or speech difficulties
- Enable real-time assessment in telemedicine

In [ ]:
# 1.1 Import Required Libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import random
from datetime import datetime
import warnings
import tqdm

# Audio processing
import librosa
import librosa.display

# Machine learning
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

# Deep learning
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Dropout, Conv1D, MaxPooling1D, GlobalMaxPooling1D
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.utils import to_categorical

# Set seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)
random.seed(42)
warnings.filterwarnings('ignore')

## 2. Data Acquisition and Inspection

We load the metadata and audio files, inspect for missing/corrupt files, and summarize the dataset.

In [ ]:
# 2.1 Define dataset paths
BASE_PATH = r"G:\Msc\NCU\Doctoral Record\multimodal_medical_diagnosis\data\Medical Speech, Transcription, and Intent"
PATHS = {
    'csv': os.path.join(BASE_PATH, "overview-of-recordings.csv"),
    'test_audio': os.path.join(BASE_PATH, "recordings", "test"),
    'train_audio': os.path.join(BASE_PATH, "recordings", "train"),
    'validate_audio': os.path.join(BASE_PATH, "recordings", "validate")
}

In [ ]:
# 2.2 Load metadata and inspect structure
metadata = pd.read_csv(PATHS['csv'])
print(f"Loaded metadata: {metadata.shape[0]} records, {metadata.shape[1]} columns")
print(f"Columns: {list(metadata.columns)}")
display(metadata.head())

In [ ]:
# 2.3 List audio files in each split directory
for split in ['train_audio', 'test_audio', 'validate_audio']:
    audio_dir = PATHS[split]
    files = [f for f in os.listdir(audio_dir) if f.endswith('.wav')]
    print(f"Found {len(files)} audio files in {audio_dir}")

In [ ]:
# 2.4 Check for missing audio files referenced in metadata
missing_files = []
for idx, row in metadata.iterrows():
    file_name = str(row['file_name'])
    found = False
    for audio_dir in [PATHS['train_audio'], PATHS['test_audio'], PATHS['validate_audio']]:
        file_path = os.path.join(audio_dir, file_name)
        if os.path.exists(file_path):
            found = True
            break
    if not found:
        missing_files.append(file_name)
print(f"Number of missing audio files referenced in metadata: {len(missing_files)}")
if missing_files:
    print("Examples of missing files:", missing_files[:5])
else:
    print("All referenced audio files are present.")

## 3. Feature Extraction

We extract Mel-frequency cepstral coefficients (MFCCs) and other spectral features from each audio file. This step is critical for transforming raw audio into a format suitable for machine learning.

In [ ]:
# 3.1 Audio feature extraction function
def extract_audio_features(file_path, sr=22050, n_mfcc=13):
    """
    Extracts MFCCs and other features from an audio file.
    Returns a 1D feature vector.
    """
    try:
        y, sr = librosa.load(file_path, sr=sr)
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
        chroma = librosa.feature.chroma_stft(y=y, sr=sr)
        zcr = librosa.feature.zero_crossing_rate(y)
        spec_centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
        spec_bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)
        features = np.hstack([
            np.mean(mfccs, axis=1),
            np.std(mfccs, axis=1),
            np.mean(chroma, axis=1),
            np.std(chroma, axis=1),
            np.mean(zcr),
            np.std(zcr),
            np.mean(spec_centroid),
            np.std(spec_centroid),
            np.mean(spec_bandwidth),
            np.std(spec_bandwidth)
        ])
        return features
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return np.zeros(13*2 + 12*2 + 6)  # Return zeros if processing fails

In [ ]:
# 3.2 Build feature matrix and label vector
feature_list = []
label_list = []
valid_files = []
for idx, row in tqdm.tqdm(metadata.iterrows(), total=len(metadata)):
    file_name = str(row['file_name'])
    for audio_dir in [PATHS['train_audio'], PATHS['test_audio'], PATHS['validate_audio']]:
        file_path = os.path.join(audio_dir, file_name)
        if os.path.exists(file_path):
            try:
                features = extract_audio_features(file_path)
                feature_list.append(features)
                label_list.append(row['prompt'])
                valid_files.append(file_path)
            except Exception as e:
                print(f"Error processing {file_path}: {e}")
            break
X = np.array(feature_list)
y = np.array(label_list)
print(f"Feature matrix shape: {X.shape}")
print(f"Labels shape: {y.shape}")

## 4. Exploratory Data Analysis (EDA)

We explore the class distribution and feature distributions to understand the dataset and identify potential issues such as class imbalance.

In [ ]:
# 4.1 Visualize class distribution
plt.figure(figsize=(10, 5))
sns.countplot(y)
plt.title('Distribution of Medical Conditions (Prompts)')
plt.xlabel('Medical Condition')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.show()

In [ ]:
# 4.2 Visualize feature distributions (MFCC means)
plt.figure(figsize=(16, 6))
for i in range(min(13, X.shape[1])):
    plt.subplot(2, 7, i+1)
    plt.hist(X[:, i], bins=20, color='skyblue')
    plt.title(f'MFCC {i+1}')
    plt.tight_layout()
plt.show()

## 5. Feature Engineering

We encode class labels and normalize features to prepare the data for modeling.

In [ ]:
# 5.1 Encode class labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
# 5.2 Normalize features
scaler = StandardScaler()
X_normalized = scaler.fit_transform(X)
print(f"Feature matrix shape: {X_normalized.shape}")
print(f"Number of classes: {len(label_encoder.classes_)}")

## 6. Model Selection and Evaluation

We compare traditional machine learning models using cross-validation and multiple metrics.

In [ ]:
# 6.1 Train and evaluate traditional ML models
def train_and_evaluate_audio_models(X, y):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    models = {
        'Logistic Regression': LogisticRegression(max_iter=1000),
        'Random Forest': RandomForestClassifier(),
        'Decision Tree': DecisionTreeClassifier(),
        'Support Vector Machine': SVC(probability=True)
    }
    results = {}
    for name, model in models.items():
        cv_scores = cross_val_score(model, X_train, y_train, cv=5)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        results[name] = {
            'Accuracy': accuracy_score(y_test, y_pred),
            'Precision': precision_score(y_test, y_pred, average='weighted', zero_division=0),
            'Recall': recall_score(y_test, y_pred, average='weighted', zero_division=0),
            'F1 Score': f1_score(y_test, y_pred, average='weighted', zero_division=0),
            'Cross-Val Score': cv_scores.mean()
        }
        print(f"\n{name} Performance:")
        for metric, value in results[name].items():
            print(f"{metric}: {value:.4f}")
    return results

audio_model_results = train_and_evaluate_audio_models(X_normalized, y_encoded)

In [ ]:
# 6.2 Visualize model performance
def visualize_audio_model_performance(results):
    metrics_df = pd.DataFrame.from_dict(results, orient='index')
    plt.figure(figsize=(12, 6))
    metrics_df.plot(kind='bar', rot=45)
    plt.title('Audio Model Performance Comparison')
    plt.xlabel('Models')
    plt.ylabel('Score')
    plt.tight_layout()
    plt.show()
visualize_audio_model_performance(audio_model_results)

## 7. Hyperparameter Optimization

We perform hyperparameter tuning for the best-performing model using cross-validation.

In [ ]:
# 7.1 Hyperparameter tuning for Logistic Regression
param_grid = {
    'C': [0.01, 0.1, 1, 10],
    'solver': ['liblinear', 'lbfgs']
}
logreg = LogisticRegression(max_iter=1000, random_state=42)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid_search = GridSearchCV(logreg, param_grid, cv=cv, scoring='f1_weighted', n_jobs=-1)
grid_search.fit(X_normalized, y_encoded)
print(f"Best parameters: {grid_search.best_params_}")
print(f"Best cross-validated F1 score: {grid_search.best_score_:.4f}")
best_logreg = grid_search.best_estimator_

In [ ]:
# 7.2 Evaluate best model and plot confusion matrix
X_train, X_test, y_train, y_test = train_test_split(X_normalized, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)
best_logreg.fit(X_train, y_train)
y_pred = best_logreg.predict(X_test)
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

## 8. Deep Learning Models

We implement and evaluate deep learning models (CNN, LSTM) for audio classification.

In [ ]:
# 8.1 Prepare data for deep learning
num_classes = len(label_encoder.classes_)
y_categorical = to_categorical(y_encoded, num_classes=num_classes)
X_train_dl, X_test_dl, y_train_dl, y_test_dl = train_test_split(
    X_normalized, y_categorical, test_size=0.2, random_state=42, stratify=y_encoded
)
# Reshape for Conv1D/LSTM
X_train_cnn = X_train_dl.reshape((X_train_dl.shape[0], X_train_dl.shape[1], 1))
X_test_cnn = X_test_dl.reshape((X_test_dl.shape[0], X_test_dl.shape[1], 1))
print(f"Training data shape: {X_train_cnn.shape}")
print(f"Test data shape: {X_test_cnn.shape}")

In [ ]:
# 8.2 CNN model for audio features
cnn_model = Sequential([
    Conv1D(64, kernel_size=3, activation='relu', input_shape=(X_train_cnn.shape[1], 1)),
    MaxPooling1D(pool_size=2),
    Dropout(0.3),
    Conv1D(128, kernel_size=3, activation='relu'),
    GlobalMaxPooling1D(),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(num_classes, activation='softmax')
])
cnn_model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
early_stopping = EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True)
model_checkpoint = ModelCheckpoint('best_audio_cnn_model.h5', monitor='val_accuracy', save_best_only=True)
history_cnn = cnn_model.fit(
    X_train_cnn, y_train_dl,
    validation_split=0.1,
    epochs=10,
    batch_size=32,
    callbacks=[early_stopping, model_checkpoint],
    verbose=1
)
cnn_score = cnn_model.evaluate(X_test_cnn, y_test_dl, verbose=0)
print(f"CNN Test accuracy: {cnn_score[1]:.4f}")

In [ ]:
# 8.3 LSTM model for audio features
lstm_model = Sequential([
    LSTM(64, input_shape=(X_train_cnn.shape[1], 1), dropout=0.2, recurrent_dropout=0.2),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(num_classes, activation='softmax')
])
lstm_model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
history_lstm = lstm_model.fit(
    X_train_cnn, y_train_dl,
    validation_split=0.1,
    epochs=10,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)
lstm_score = lstm_model.evaluate(X_test_cnn, y_test_dl, verbose=0)
print(f"LSTM Test accuracy: {lstm_score[1]:.4f}")

In [ ]:
# 8.4 Visualize training history for CNN
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history_cnn.history['accuracy'], label='Training Accuracy')
plt.plot(history_cnn.history['val_accuracy'], label='Validation Accuracy')
plt.title('CNN Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.subplot(1, 2, 2)
plt.plot(history_cnn.history['loss'], label='Training Loss')
plt.plot(history_cnn.history['val_loss'], label='Validation Loss')
plt.title('CNN Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.tight_layout()
plt.show()

## 9. Model Interpretability

We analyze feature importance and visualize the most influential features for model predictions.

In [ ]:
# 9.1 Feature importance for engineered features model
rf_feat = RandomForestClassifier(n_estimators=100, random_state=42)
X_train_feat, X_test_feat, y_train_feat, y_test_feat = train_test_split(
    X_normalized, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)
rf_feat.fit(X_train_feat, y_train_feat)
importances = rf_feat.feature_importances_
indices = np.argsort(importances)[-15:]
plt.figure(figsize=(10, 8))
plt.barh(range(len(indices)), importances[indices], align='center')
plt.yticks(range(len(indices)), [f'Feature {i}' for i in indices])
plt.title('Feature Importance (Random Forest)')
plt.xlabel('Relative Importance')
plt.tight_layout()
plt.show()

## 10. Research Hypothesis Validation

We validate the research hypotheses based on model performance, using a clinical threshold for precision and recall.

In [ ]:
# 10.1 Hypothesis validation function
def validate_audio_research_hypothesis(results, threshold=0.75):
    print("\n--- Research Hypothesis Validation ---")
    hypothesis_met = any(
        metrics['Precision'] > threshold and metrics['Recall'] > threshold
        for metrics in results.values()
    )
    if hypothesis_met:
        print("H2a Supported: Audio analysis provides sufficient precision and recall")
        print("Recommendation: Suitable for provider decision support")
    else:
        print("H20 Supported: Audio analysis provides insufficient precision and recall")
        print("Recommendation: Further model refinement needed")
validate_audio_research_hypothesis(audio_model_results)

## 11. Limitations and Future Work

- Limited to audio-based classification; multimodal integration is a future direction
- Potential dataset bias and class imbalance
- Future work: integrate text and audio, explore advanced deep learning architectures, and collect more diverse medical audio data

## 12. Conclusion

This notebook provides a comprehensive, reproducible workflow for medical audio classification, mirroring the structure and rigor of the text classification approach. The results support the use of deep learning and ML for provider decision support based on audio data.